# Anthropic Claude on Amazon Bedrock — Messages API core

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Claude on the `bedrock-mantle` endpoint speaks the **Anthropic-native Messages
API**. This notebook covers the fundamentals; `02-thinking-tools-and-caching.ipynb`
covers extended thinking, tool use, and prompt caching, and
`03-agentic-computer-use-and-memory.ipynb` covers the agentic tool families.

**Models covered**

| Model ID | Notes |
|---|---|
| `anthropic.claude-opus-5` | Frontier |
| `anthropic.claude-sonnet-5` | Balanced frontier |
| `anthropic.claude-opus-4-8` | Previous-generation Opus |
| `anthropic.claude-opus-4-7` | Earlier Opus |
| `anthropic.claude-haiku-4-5` | Fast and cheap; different parameter support |
| `anthropic.claude-fable-5` | Specialised variant (may be gated by data-retention mode — see §4b) |
| `anthropic.claude-fable-5-1` | Newer Fable. On `bedrock-runtime` only — the endpoint check below and §4b show what mantle answers for it |

## Which API? Messages — and *only* Messages
On `bedrock-mantle`, Claude models do **not** serve the Responses API or Chat
Completions. Both return 400. We prove that in §2.

## Check the Region before you commit to one
Which Claude models are available differs by Region and changes over time, so treat
any list as a snapshot. This notebook pins `us-east-1`, where the widest selection
was available when it was written. §11 enumerates `GET /v1/models` per Region so you
can see the current picture for yourself.

## Self-contained, but see also
- **Auth (SigV4 (AWS Signature Version 4) + short-term keys), the three URL paths,
  model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects/Workspaces, cost attribution, data retention / ZDR (zero data retention),
  CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retry/backoff, service tiers, TTFT (time-to-first-token)** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs anthropic, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `stream_lines` | raw SSE lines from a streaming endpoint, no SDK |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `converse_tool_uses` | the `toolUse` blocks from a Converse response |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, err, keyword_recall, post, safe_print, slide_jpeg

REGION = "us-east-1"  # full Claude set as of Aug 2026; verify per Region

OPUS5 = "anthropic.claude-opus-5"
SONNET5 = "anthropic.claude-sonnet-5"
OPUS48 = "anthropic.claude-opus-4-8"
HAIKU45 = "anthropic.claude-haiku-4-5"
OPUS47 = "anthropic.claude-opus-4-7"

# Claude has its own path prefix on mantle — neither /v1 nor /openai/v1.
PREFIX = "/anthropic/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws/anthropic"
print("base URL:", BASE_URL)

# The anthropic-version header is REQUIRED on every mantle Messages request.
# (On bedrock-runtime the equivalent goes in the body as "anthropic_version".)
AV = {"anthropic-version": "2023-06-01"}


def claude_text(payload: dict) -> str:
    """Concatenate the TEXT blocks of a Messages response, ignoring thinking blocks.

    Never index content[0] blindly: reasoning-capable models put a `thinking`
    block first. Section 2b demonstrates this.
    """
    return "".join(
        b.get("text", "") for b in payload.get("content", []) if b.get("type") == "text"
    )

base URL: https://bedrock-mantle.us-east-1.api.aws/anthropic


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one". The
table also has the other direction, and a mantle notebook has to say so: a model
the mantle catalogue does not list cannot be called from any cell here at all.

In [2]:
from bedrock import endpoints_for, list_models, runtime_id_for

COVERED = [
    "anthropic.claude-fable-5",
    "anthropic.claude-fable-5-1",
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-4-7",
    "anthropic.claude-opus-4-8",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
]

# endpoints_for() answers mantle=False both when a model is absent and when the
# mantle catalogue cannot be READ at all, and those must not reach the same verdict.
# On a test host without bedrock-mantle list permission every row below landed in the
# "runtime only" bucket, which reads as a finding about Claude rather than about the
# host. So ask for the catalogue separately and let its absence disable the verdict.
try:
    mantle_catalogue = set(list_models(REGION))
except (RuntimeError, OSError) as exc:
    mantle_catalogue = None
    print(f"!! bedrock-mantle catalogue unreachable in {REGION}: {type(exc).__name__}")

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
# Three buckets, not two. COVERED used to hold only models mantle serves, so the cell
# could print "mantle only" or "both" and nothing else -- and it printed "both" for
# anything with a runtime id, without asking whether mantle had one. fable-5-1 is on
# runtime and NOT in the mantle catalogue, which is a case a mantle notebook needs
# to name: section 4b shows how mantle refuses it.
mantle_only, runtime_only, neither = [], [], []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if mantle_catalogue is None:
        label = f"{label or 'runtime'} (mantle unknown)"
    elif where["mantle"] and not where["runtime"]:
        mantle_only.append(model_id)
    elif where["runtime"] and not where["mantle"]:
        runtime_only.append(model_id)
    elif not (where["mantle"] or where["runtime"]):
        neither.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
# The raw mantle catalogue is a third source to check the second against. COVERED
# holds mantle names, so exact membership there must agree with endpoints_for()'s
# normalised answer; where it does not, one of the two is keyed wrong.
mantle_disagree = [] if mantle_catalogue is None else [
    m for m in COVERED
    if (m in mantle_catalogue) != endpoints_for(m, REGION)["mantle"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
if mantle_disagree:
    print(f"!! endpoints_for() and the mantle catalogue DISAGREE for {mantle_disagree}.")
    print("   One of the two is keyed wrong in one direction or the other, so read")
    print("   list_models(REGION) directly before trusting those rows.")
on_runtime = [m for m in COVERED if runtime_id_for(m, REGION) is not None]
print(f"=> {len(on_runtime)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_catalogue is None:
    print("   The mantle half of every row is UNKNOWN, not False, so this cell has")
    print("   no verdict on which endpoint serves what. Grant the mantle catalogue")
    print("   permission listed in the README and re-run before you rely on it.")
else:
    if mantle_only:
        print(f"   bedrock-mantle only: {mantle_only}")
        print(f"   Of the two endpoints, mantle is the one serving those in {REGION}.")
    if runtime_only:
        print(f"   bedrock-runtime only: {runtime_only}")
        print("   The mantle catalogue does not list those, so no cell below reaches")
        print("   them from this endpoint; section 4b shows the 404 mantle answers,")
        print("   and the runtime id above is what does reach them.")
    if neither:
        print(f"   on neither endpoint in {REGION}: {neither}")
    if not (mantle_only or runtime_only or neither):
        print(f"   Both endpoints serve each model above in {REGION} on this run.")
        print("   This notebook shows the bedrock-mantle calls; the ids above are")
        print("   what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------
anthropic.claude-fable-5               us.anthropic.claude-fable-5              mantle, runtime
anthropic.claude-fable-5-1             us.anthropic.claude-fable-5-1            runtime
anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime
anthropic.claude-opus-4-7              us.anthropic.claude-opus-4-7             mantle, runtime
anthropic.claude-opus-4-8              us.anthropic.claude-opus-4-8             mantle, runtime
anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime
anthropic.claude-sonnet-5              us.anthropic.claude-sonnet-5             mantle, runtime

=> 7/7 of these are on bedrock-runtime; 7 under a different id.
   bedrock-runtime only: ['anthropic.claude-fable-5-1']
   The mantle catalogue d

## 1. First call

Auth is a short-term Bedrock API key minted from ambient IAM credentials —
valid ≤12 h, not refreshable, Region-pinned. (`../00-foundations/01` shows the
self-refreshing provider and the SigV4 alternative.)

In [3]:
code, data = post(
    f"{PREFIX}/messages",
    {
        "model": SONNET5,
        "max_tokens": 300,  # REQUIRED on the Messages API, unlike OpenAI APIs
        "messages": [
            {
                "role": "user",
                "content": ("Explain idempotency in two sentences."),
            }
        ],
    },
    region=REGION,
    headers=AV,
)
print("HTTP", code)
print(claude_text(data))
print("\nusage:", json.dumps(data["usage"]))
print("stop_reason:", data["stop_reason"])

HTTP 200
Idempotency is a property where performing an operation multiple times produces the same result as performing it once, regardless of how many times it's repeated. This is especially important in APIs and distributed systems, where retrying a request (due to network issues, timeouts, etc.) shouldn't cause unintended side effects like duplicate charges or repeated data creation.

usage: {"input_tokens": 19, "cache_creation_input_tokens": 0, "cache_read_input_tokens": 0, "cache_creation": {"ephemeral_5m_input_tokens": 0, "ephemeral_1h_input_tokens": 0}, "output_tokens": 110, "output_tokens_details": {"thinking_tokens": 0}, "service_tier": "standard"}
stop_reason: end_turn


Note `max_tokens` is **mandatory** here. Omit it and the request is rejected —
the OpenAI-compatible APIs default it for you, Messages does not.

In [4]:
code, data = post(
    f"{PREFIX}/messages",
    {"model": SONNET5, "messages": [{"role": "user", "content": "Hi"}]},
    region=REGION,
    headers=AV,
)
print(f"omitting max_tokens -> HTTP {code}: {err(data)[:110]}")

omitting max_tokens -> HTTP 400: max_tokens: Field required


## 2. Why Messages and not the OpenAI-compatible APIs

Probe all three surfaces so the 400s are visible rather than asserted:

In [5]:
for label, path, body in [
    (
        "Messages",
        f"{PREFIX}/messages",
        {
            "model": SONNET5,
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Reply OK"}],
        },
    ),
    (
        "Responses /v1",
        "/v1/responses",
        {"model": SONNET5, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "Responses /openai/v1",
        "/openai/v1/responses",
        {"model": SONNET5, "input": "Reply OK", "max_output_tokens": 16},
    ),
    (
        "ChatCompletions /v1",
        "/v1/chat/completions",
        {
            "model": SONNET5,
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
    ),
]:
    code, resp = post(path, body, region=REGION, headers=AV)
    print(f"  {label:22} -> HTTP {code} {'' if code == 200 else err(resp)[:60]}")

  Messages               -> HTTP 200 


  Responses /v1          -> HTTP 400 The model 'anthropic.claude-sonnet-5' does not support the '


  Responses /openai/v1   -> HTTP 400 The model 'anthropic.claude-sonnet-5' does not support the '


  ChatCompletions /v1    -> HTTP 400 The model 'anthropic.claude-sonnet-5' does not support the '


So on mantle, Claude is Messages-only. Practical consequences:

- No `previous_response_id`; you manage conversation history yourself.
- Structured output uses tools, not `response_format` / `text.format` (§7).
- Cost attribution uses the `anthropic-workspace` header, not `OpenAI-Project`.

## 2b. Read the content array properly — never assume `content[0]` is text

Reasoning-capable Claude models return a **`thinking` block before the text
block**. `content[0].text` therefore raises or returns nothing on those models.
Always filter by block type.

In [6]:
for model in (SONNET5, OPUS5):
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            # Budget generously. At 200 tokens a reasoning model spends the lot on
            # thinking and returns no text block at all, so the "safe read" below
            # printed '' and the section demonstrated only half its point.
            "max_tokens": 1500,
            "messages": [
                {
                    "role": "user",
                    "content": "What problem does a write-ahead log solve?",
                }
            ],
        },
        region=REGION,
        headers=AV,
    )
    kinds = [b.get("type") for b in data.get("content", [])]
    print(f"{model:30} blocks={kinds}")
    print(f"    text via helper: {claude_text(data).strip()[:80]!r}")
    first = data["content"][0]
    naive_verdict = "work" if first.get("type") == "text" else "FAIL"
    print(
        f"    content[0]['type'] = {first.get('type')!r} "
        f"-> naive content[0]['text'] would {naive_verdict}"
    )

anthropic.claude-sonnet-5      blocks=['text']
    text via helper: '# Write-Ahead Log (WAL)\n\nA write-ahead log solves the **durability and crash rec'
    content[0]['type'] = 'text' -> naive content[0]['text'] would work


anthropic.claude-opus-5        blocks=['thinking', 'text']
    text via helper: "## The core problem\n\nStorage devices don't give you atomicity across the units y"
    content[0]['type'] = 'thinking' -> naive content[0]['text'] would FAIL


## 3. The Anthropic SDK

Point the official SDK at mantle by overriding `base_url`. Note the base URL
omits the `/v1` — the SDK appends it.

In [7]:
import anthropic
from aws_bedrock_token_generator import provide_token

client = anthropic.Anthropic(
    api_key=provide_token(region=REGION),  # fresh token; do not cache for hours
    base_url=BASE_URL,
)

message = client.messages.create(
    model=SONNET5,
    max_tokens=250,
    system="You are terse and precise.",
    messages=[
        {
            "role": "user",
            "content": ("What problem does a write-ahead log solve?"),
        }
    ],
)
# Filter by block type rather than indexing [0] — see section 2b.
print("".join(b.text for b in message.content if b.type == "text"))
print("\nmodel echoed:", message.model, "| stop:", message.stop_reason)

## The Problem

Databases and storage systems need to keep data durable and consistent, but writing changes directly to their final on-disk data structures (B-trees, heaps, etc.) on every update is dangerous and slow:

1. **Crash consistency** — If a crash happens mid-write, in-place updates can leave data structures partially modified/corrupted, with no way to know what completed.
2. **Durability vs. performance** — Flushing complex, scattered data structure pages to disk on every transaction is slow (random I/O), but you still need a guarantee that committed changes survive a crash.
3. **Atomicity of multi-step changes** — Some operations touch multiple pages/structures; a crash partway through leaves them inconsistent.

## The Solution

A **write-ah

model echoed: claude-sonnet-5 | stop: max_tokens


## 4. `temperature` is deprecated on newer Claude models

This is the parameter trap for this family. Frontier models reject `temperature`
outright; `haiku-4-5` still accepts it. Never share a sampling config across
Claude generations.

In [8]:
print(f"{'model':32} {'temperature=0.5':>16}")
print("-" * 50)
for model in (OPUS5, SONNET5, OPUS48, OPUS47, HAIKU45):
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 16,
            "temperature": 0.5,
            "messages": [{"role": "user", "content": "Reply OK"}],
        },
        region=REGION,
        headers=AV,
    )
    verdict = "accepted" if code == 200 else f"{code}: {err(data)[:40]}"
    print(f"{model:32} {verdict:>16}")

model                             temperature=0.5
--------------------------------------------------


anthropic.claude-opus-5          400: `temperature` is deprecated for this mod


anthropic.claude-sonnet-5        400: `temperature` is deprecated for this mod


anthropic.claude-opus-4-8        400: `temperature` is deprecated for this mod


anthropic.claude-opus-4-7        400: `temperature` is deprecated for this mod


anthropic.claude-haiku-4-5               accepted


The fix is to omit it — the newer models are tuned to run without
sampling knobs.

In [9]:
for model in (OPUS5, HAIKU45):
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": 60,
            "messages": [{"role": "user", "content": "One word: capital of Japan?"}],
        },
        region=REGION,
        headers=AV,
    )
    print(f"{model:32} HTTP {code} -> {claude_text(data).strip()[:40]!r}")

anthropic.claude-opus-5          HTTP 200 -> 'Tokyo'


anthropic.claude-haiku-4-5       HTTP 200 -> 'Tokyo'


## 4b. A model can be gated by your data-retention mode

Some models are only offered under specific data-retention modes. If your
account's mode is not in the model's `allowed_modes`, inference returns 400 and
`GET /v1/models/{id}` reports `status: unavailable` with a reason. Check before
you debug your request shape.

In [10]:
code, retention = post("/v1/data_retention", None, region=REGION, method="GET")
print("account data_retention:", retention)

# Fable 5.1 is here too: it is the newer frontier model, and it carries the same
# Covered Model designation as Fable 5, which is what this cell is really about.
blocked = []
for mid in (SONNET5, "anthropic.claude-fable-5", "anthropic.claude-fable-5-1"):
    code, info = post(f"/v1/models/{mid}", None, region=REGION, method="GET")
    if code != 200:
        print(f"\n{mid}\n   GET /v1/models -> {code}: {err(info)[:90]}")
        continue
    dr = info.get("data_retention", {})
    allowed = dr.get("allowed_modes")
    print(f"\n{mid}")
    print(f"   status       : {info.get('status')}")
    print(f"   allowed_modes: {allowed}")
    if info.get("status_reason"):
        print(f"   reason       : {info['status_reason'][:110]}")
    if allowed is not None and "default" not in allowed:
        blocked.append((mid, allowed))

# Derive the consequence rather than describing it.
account_mode = (retention or {}).get("mode")
print(f"\n=> account retention mode: {account_mode!r}")
if blocked:
    print(f"=> {len(blocked)} model(s) here do NOT permit 'default', so they are")
    print("   unavailable while the account resolves to it:")
    for mid, allowed in blocked:
        print(f"     {mid:34} permits only {allowed}")
    print("   These are Anthropic Covered Models. The designation brings extra data")
    print("   retention, safety-review and access policy, which is why the mode you")
    print("   are on decides whether you can call them at all -- the 400 is a policy")
    print("   answer, not a capacity or Region problem.")
    print("   Note the endpoint asymmetry: the SAME models answer 200 on")
    print("   bedrock-runtime via a geo profile. A refusal here is not 'unreachable'.")
else:
    print("=> every model above permits 'default' on this account.")

account data_retention: {'mode': 'inherit'}



anthropic.claude-sonnet-5
   status       : available
   allowed_modes: ['aws_review', 'none', 'provider_data_share', 'default']



anthropic.claude-fable-5
   status       : unavailable
   allowed_modes: ['provider_data_share', 'aws_review']
   reason       : This model is not available under data retention mode 'default'.



anthropic.claude-fable-5-1
   GET /v1/models -> 404: The model 'anthropic.claude-fable-5-1' does not exist

=> account retention mode: 'inherit'
=> 1 model(s) here do NOT permit 'default', so they are
   unavailable while the account resolves to it:
     anthropic.claude-fable-5           permits only ['provider_data_share', 'aws_review']
   These are Anthropic Covered Models. The designation brings extra data
   retention, safety-review and access policy, which is why the mode you
   are on decides whether you can call them at all -- the 400 is a policy
   answer, not a capacity or Region problem.
   Note the endpoint asymmetry: the SAME models answer 200 on
   bedrock-runtime via a geo profile. A refusal here is not 'unreachable'.


`allowed_modes` lives **inside `data_retention`**, and the cell above returns it for
every model it asks about — so read it from there rather than from the top level of the
payload, and do not treat its absence as "no restrictions". This paragraph used to say
it "is returned for some models", which its own output contradicts.

Two things worth carrying away:

- **A retention mode is an availability control, not just a compliance setting.** A
  Covered Model that permits only `provider_data_share` is `unavailable` while your
  account resolves to `default`, and the 400 says so in those words. See
  `../00-foundations/02` for how project, account and model default combine.
- **A refusal is endpoint-specific.** The same Fable models that 400 here answer 200 on
  `bedrock-runtime` addressed with a geo profile. This collection previously recorded
  Fable as "cannot be probed from this account" on the strength of the mantle 400 alone;
  that was a conclusion drawn from one endpoint and stated as a property of the model.

## 5. Streaming

Messages streams **named SSE (server-sent events) events** — `message_start`,
`content_block_delta`,
`message_stop` — which is a different shape from the OpenAI APIs' `data: {…}`
frames.

In [11]:
event_counts = {}
print("--- live stream ---")
try:
    with client.messages.stream(
        model=SONNET5,
        max_tokens=300,
        messages=[{"role": "user", "content": "List three properties of a good API."}],
    ) as stream:
        for event in stream:
            event_counts[event.type] = event_counts.get(event.type, 0) + 1
            if event.type == "text":
                print(event.text, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream 5xx is
    # not rare, and it has happened while building these notebooks. Report what
    # arrived instead of losing it - production code has to decide whether a
    # partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after {sum(event_counts.values())} events: "
          f"{type(exc).__name__}]")

print("\n\n--- event types ---")
for name, count in sorted(event_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")


--- live stream ---


# Three Properties of a

 Good API

1. **

Consistency** – A

 good API foll

...

 notable properties wor

th mentioning:**


- **Simplicity/

Ease of



--- event types ---
    63  content_block_delta
    63  text
     1  message_start
     1  content_block_start
     1  content_block_stop
     1  message_delta
     1  message_stop


In [12]:
# The raw SSE, for anyone not using the SDK.
from bedrock import stream_lines

shown = 0
try:
    for line in stream_lines(
        f"{PREFIX}/messages",
        {
            "model": HAIKU45,
            "max_tokens": 40,
            "stream": True,
            "messages": [{"role": "user", "content": "Count to three."}],
        },
        region=REGION,
        headers=AV,
    ):
        print(line[:110])
        shown += 1
        if shown >= 8:
            print("…")
            break
except Exception as exc:
    # Same discipline as the SDK path above: a raw SSE stream can also drop
    # mid-flight, and the lines already printed are still valid output.
    print(f"[stream interrupted after {shown} lines: {type(exc).__name__}]")


event: message_start
data: {"type":"message_start","message":{"model":"claude-haiku-4-5-20251001","id":"msg_bdrk_dz674au7lg53arswq3
event: content_block_start
data: {"type":"content_block_start","index":0,"content_block":{"type":"text","text":""}}
event: content_block_delta
data: {"type":"content_block_delta","index":0,"delta":{"type":"text_delta","text":"One."}}
event: content_block_delta
data: {"type":"content_block_delta","index":0,"delta":{"type":"text_delta","text":"\n\nTwo.\n\nThree."}}
…


## 6. Multi-turn, system prompts, and prefill

Messages must alternate user/assistant and **start with `user`**. The system
prompt is a separate top-level parameter, not a message.

In [13]:
history = [{"role": "user", "content": "Name a NoSQL database."}]
first = client.messages.create(
    model=HAIKU45,
    max_tokens=100,
    system="Answer with the name only, no punctuation.",
    messages=history,
)
answer = "".join(b.text for b in first.content if b.type == "text").strip()
print("assistant:", answer)

history += [
    {"role": "assistant", "content": answer},
    {
        "role": "user",
        "content": "Now name its main consistency trade-off in one sentence.",
    },
]
second = client.messages.create(model=HAIKU45, max_tokens=150, messages=history)
print("assistant:", "".join(b.text for b in second.content if b.type == "text").strip())
print(f"\ninput tokens grew: {first.usage.input_tokens} -> {second.usage.input_tokens}")

assistant: MongoDB


assistant: MongoDB trades immediate consistency for availability and partition tolerance, meaning writes may not be instantly visible across all replicas depending on the write concern level used.

input tokens grew: 23 -> 32


In [14]:
# Prefill: seed the assistant turn to constrain the format. Claude continues from
# your partial text — very effective for forcing JSON or a fixed prefix.
prefilled = client.messages.create(
    model=HAIKU45,
    max_tokens=150,
    messages=[
        {"role": "user", "content": "Give three AWS Regions in the US."},
        {"role": "assistant", "content": "1. us-east-1"},  # prefill
    ],
)
print(
    "continuation:",
    "".join(b.text for b in prefilled.content if b.type == "text")[:150],
)

continuation:  (N. Virginia)
2. us-west-2 (Oregon)
3. us-west-1 (N. California)


## 7. Structured output — prefer a forced tool

The Messages API has an `output_config.format` structured-output parameter. Whether
a given endpoint and model accept it is worth checking rather than assuming — the
cell below does exactly that, and prints what you get today.

Forcing a tool works regardless, and the arguments *are* your JSON, so it is the
portable choice either way.

In [15]:
code, data = post(
    f"{PREFIX}/messages",
    {
        "model": SONNET5,
        "max_tokens": 200,
        "messages": [{"role": "user", "content": "Describe France."}],
        "output_config": {
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {"capital": {"type": "string"}},
                },
            }
        },
    },
    region=REGION,
    headers=AV,
)
print(f"output_config.format -> HTTP {code}: {err(data)[:120]}")
print("\n=> For structured outputs with Claude either force a tool (below),")
print("   or use the Converse API on the bedrock-runtime endpoint.")

output_config.format -> HTTP 400: output_config.format: Extra inputs are not permitted

=> For structured outputs with Claude either force a tool (below),
   or use the Converse API on the bedrock-runtime endpoint.


In [16]:
# The portable route: a forced tool whose input schema IS your output schema.
country_tool = {
    "name": "emit_country",
    "description": "Return structured facts about a country.",
    "input_schema": {
        "type": "object",
        "properties": {
            "country": {"type": "string"},
            "capital": {"type": "string"},
            "population_millions": {"type": "number"},
        },
        "required": ["country", "capital", "population_millions"],
    },
}

structured = client.messages.create(
    model=SONNET5,
    max_tokens=400,
    tools=[country_tool],
    tool_choice={"type": "tool", "name": "emit_country"},  # force it
    messages=[{"role": "user", "content": "Describe France."}],
)
tool_blocks = [b for b in structured.content if b.type == "tool_use"]
print("stop_reason:", structured.stop_reason)
print(json.dumps(tool_blocks[0].input, indent=2))

stop_reason: tool_use
{
  "country": "France",
  "capital": "Paris",
  "population_millions": 67.4
}


Because the schema is enforced at the tool-call layer, the result is already a
dict — no string parsing, and no risk of trailing characters breaking a JSON
parse (a real problem with some other mantle families).

## 8. Count tokens before you spend them

`count_tokens` on the Messages API is a control-plane call that prices a request
without running it — useful for pre-flight cost estimation and for deciding whether
a prompt fits.

**It exists on both endpoints, with different coverage**, which is worth knowing
before you conclude it is missing:

| | `bedrock-mantle` | `bedrock-runtime` |
|---|---|---|
| How | `POST /anthropic/v1/messages/count_tokens` | `bedrock-runtime` `count_tokens` API |
| Claude 5 generation (`opus-5`, `sonnet-5`, `opus-4-8`, `opus-4-7`, `fable-5`) | **works** | **refused** — *"model doesn't support counting tokens"* |
| Claude 4.x (`haiku-4-5`, `opus-4-6`, `opus-4-5`, `opus-4-1`) | works | **works**, with the bare versioned ID |
| Inference profiles (`us.` prefix) | n/a | **refused** |
| Other providers (Nova, Llama, GPT, Gemma…) | n/a | refused |

So on `bedrock-runtime` it is a Claude-4.x-only facility that will not take the
very profile ID Converse requires elsewhere. §8b shows that. On mantle it covers
the current generation, which is usually the one you want.

In [17]:
for label, text in [
    ("short", "Hello world"),
    ("medium", "Explain distributed consensus. " * 20),
    ("long", "Explain distributed consensus. " * 200),
]:
    code, data = post(
        f"{PREFIX}/messages/count_tokens",
        {"model": OPUS5, "messages": [{"role": "user", "content": text}]},
        region=REGION,
        headers=AV,
    )
    print(f"  {label:8} chars={len(text):6} -> input_tokens={data.get('input_tokens')}")

  short    chars=    11 -> input_tokens=10


  medium   chars=   620 -> input_tokens=206


  long     chars=  6200 -> input_tokens=2006


In [18]:
# It also counts system prompts and tool definitions, which is where budgets
# usually go missing.
code, bare = post(
    f"{PREFIX}/messages/count_tokens",
    {"model": OPUS5, "messages": [{"role": "user", "content": "Hi"}]},
    region=REGION,
    headers=AV,
)
code, with_extras = post(
    f"{PREFIX}/messages/count_tokens",
    {
        "model": OPUS5,
        "system": "You are a meticulous assistant that always cites sources.",
        "tools": [country_tool],
        "messages": [{"role": "user", "content": "Hi"}],
    },
    region=REGION,
    headers=AV,
)
print(f"bare prompt          : {bare['input_tokens']} tokens")
print(f"+ system + 1 tool def: {with_extras['input_tokens']} tokens")
print(
    f"overhead             : {with_extras['input_tokens'] - bare['input_tokens']} "
    "tokens"
)

bare prompt          : 9 tokens
+ system + 1 tool def: 423 tokens
overhead             : 414 tokens


### 8b. The same thing on `bedrock-runtime`, and its two refusals

`bedrock-runtime` exposes `CountTokens` as a first-class operation. Two things about
it catch people out, and both are visible below:

- It **refuses an inference profile**. Every other `bedrock-runtime` call for a
  profile-only model wants the `us.` form; this one wants the bare versioned ID.
- It **refuses the Claude 5 generation** outright, so the models you are most
  likely to be running are the ones it does not price.

In [19]:
from bedrock import resolve_runtime_id, runtime_client

runtime = runtime_client(REGION)
MESSAGE = {"converse": {"messages": [
    {"role": "user", "content": [{"text": "How many tokens is this sentence?"}]}
]}}

print(f"{'model id sent':46} {'result':>34}")
print("-" * 82)
for label in (
    "anthropic.claude-haiku-4-5-20251001-v1:0",   # Claude 4.x, bare + version
    "anthropic.claude-opus-4-6-v1",               # 4.x, no :0 suffix
    resolve_runtime_id(SONNET5, REGION),          # what Converse wants: us. profile
    SONNET5,                                      # Claude 5, bare
):
    try:
        answer = runtime.count_tokens(modelId=label, input=MESSAGE)
        print(f"{label:46} {str(answer['inputTokens']) + ' tokens':>34}")
    except Exception as exc:  # noqa: BLE001 - the refusal IS the lesson
        print(f"{label:46} {str(exc).split(': ')[-1][:34]:>34}")

print()
print("=> Pre-flight pricing on bedrock-runtime is a Claude 4.x facility, and it")
print("   wants the bare versioned ID rather than the inference profile. For the")
print("   Claude 5 generation, use the mantle count_tokens above instead.")

model id sent                                                              result
----------------------------------------------------------------------------------


anthropic.claude-haiku-4-5-20251001-v1:0                                30 tokens


anthropic.claude-opus-4-6-v1                                            31 tokens


us.anthropic.claude-sonnet-5                   The provided model doesn't support


anthropic.claude-sonnet-5                      The provided model doesn't support

=> Pre-flight pricing on bedrock-runtime is a Claude 4.x facility, and it
   wants the bare versioned ID rather than the inference profile. For the
   Claude 5 generation, use the mantle count_tokens above instead.


## 9. Vision

Images go in the `content` array as `source` blocks (base64 or URL).

In [20]:
import base64

# A slide from a public AWS talk, so the answer is checkable. See
# ../_shared/bedrock.py for provenance.
slide_b64 = base64.b64encode(slide_jpeg()).decode()

vision = client.messages.create(
    model=SONNET5,
    max_tokens=260,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": slide_b64,
                    },
                },
                {
                    "type": "text",
                    "text": "Read this slide. Give its title, then quote "
                            "the three green callout lines.",
                },
            ],
        }
    ],
)
answer = "".join(b.text for b in vision.content if b.type == "text").strip()
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
print(f"callouts {hits}/{total}, title={SLIDE_TITLE.lower() in answer.lower()}")
print(" ", " ".join(answer.split())[:200])

callouts 3/3, title=True
  **Title:** Ingestion from database **Green callout lines:** 1. ✓ Pay according to job duration 2. ✓ Lower compute cost, up to 90% 3. ✓ Lower storage cost for rarely accessed data


## 10. Compare the models

Same prompt across generations. Note we omit `temperature` so the call is valid
on every model (see §4).

In [21]:
task = "In one sentence, why is exactly-once delivery hard in distributed systems?"

# 900, not 160. At 160 `claude-opus-5` spent the whole budget before emitting
# any text and this table published an empty answer beside it, which reads as a
# broken model rather than as a budget too small for a reasoning-first Claude.
# `stop_reason` is printed so a future truncation says so instead of hiding.
BUDGET = 900
print(f"{'model':34} {'latency':>9} {'in':>5} {'out':>5} {'stop':>10}  answer")
print("-" * 124)
for model in (HAIKU45, OPUS47, OPUS48, SONNET5, OPUS5):
    started = time.perf_counter()
    code, data = post(
        f"{PREFIX}/messages",
        {
            "model": model,
            "max_tokens": BUDGET,
            "messages": [{"role": "user", "content": task}],
        },
        region=REGION,
        headers=AV,
    )
    elapsed = time.perf_counter() - started
    if code != 200:
        print(f"{model:34} {'-':>9} {'-':>5} {'-':>5} {'-':>10}  "
              f"HTTP {code}: {err(data)[:40]}")
        continue
    text = " ".join(claude_text(data).split())
    usage = data["usage"]
    stop = data.get("stop_reason")
    flag = "" if stop == "end_turn" else " !!"
    print(
        f"{model:34} {elapsed:>8.2f}s {usage['input_tokens']:>5} "
        f"{usage['output_tokens']:>5} {str(stop):>10}  {text[:46]!r}{flag}"
    )
    if not text:
        print(f"{'':34} {'':>36}  ^ empty: the budget went to this "
              f"model's thinking before any\n{'':34} {'':>36}    text was emitted. Raise BUDGET.")

model                                latency    in   out       stop  answer
----------------------------------------------------------------------------------------------------------------------------


anthropic.claude-haiku-4-5             1.61s    22    46   end_turn  'Exactly-once delivery is hard because you must'


anthropic.claude-opus-4-7              3.24s    36   118   end_turn  'Exactly-once delivery is hard because network '


anthropic.claude-opus-4-8              3.82s    31   112   end_turn  'Exactly-once delivery is hard because network '


anthropic.claude-sonnet-5              3.31s    31   136   end_turn  'Exactly-once delivery is hard because networks'


anthropic.claude-opus-5                5.60s    31   298   end_turn  'Because a sender can never distinguish a lost '


## 11. Check the Region yourself

Which Claude models exist in which Region changes over time, so the useful thing is
the check, not the answer. The cell below enumerates the models per Region so you
can see today's picture before committing to one.

In [22]:
from bedrock import list_models

print(f"{'region':14} {'anthropic models':>17}")
print("-" * 34)
for region in ("us-east-1", "us-east-2", "us-west-2", "eu-central-1"):
    try:
        claude = [m for m in list_models(region) if m.startswith("anthropic.")]
    except (RuntimeError, OSError) as exc:
        print(f"{region:14} error: {type(exc).__name__}")
        continue
    short_names = ", ".join(c.split(".")[-1] for c in claude)
    print(f"{region:14} {len(claude):>17}  {short_names or '(none)'}")

region          anthropic models
----------------------------------


us-east-1                      6  claude-fable-5, claude-haiku-4-5, claude-opus-4-7, claude-opus-4-8, claude-opus-5, claude-sonnet-5


us-east-2                      1  claude-haiku-4-5


us-west-2                      1  claude-haiku-4-5


eu-central-1                   0  (none)


## 12. Cost attribution with Workspaces

Same underlying resource as OpenAI-API "Projects", but referenced with the
`anthropic-workspace` header. (Details: `../00-foundations/02`.)

In [23]:
code, workspace = post(
    "/v1/organization/projects",  # control plane is always /v1
    {
        "name": "claude-samples",
        "tags": {"Application": "ClaudeDemo", "Environment": "Demo"},
    },
    region=REGION,
)
workspace_id = workspace.get("id")
safe_print("workspace:", code, workspace_id)

code, data = post(
    f"{PREFIX}/messages",
    {
        "model": HAIKU45,
        "max_tokens": 16,
        "messages": [{"role": "user", "content": "Reply OK"}],
    },
    region=REGION,
    headers={**AV, "anthropic-workspace": workspace_id},
)
print("attributed call ->", code)

code, archived = post(
    f"/v1/organization/projects/{workspace_id}/archive", {}, region=REGION
)
print("archived:", code, archived.get("status"))

workspace: 200 proj_owa57uii...


attributed call -> 200


archived: 200 archived


## Gotchas — Claude on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Messages only | Responses **and** Chat Completions both 400 on mantle |
| Path prefix | `/anthropic/v1`, not `/v1` or `/openai/v1` |
| `anthropic-version` | Required **header** on mantle (body field on runtime) |
| `max_tokens` | Mandatory — omitting it is a 400 |
| `temperature` | Acceptance is per model and changes — probe it (§4) before sending it |
| `output_config.format` | Acceptance varies — probe it (§7); a forced tool is portable |
| Region | Availability differs per Region and changes — enumerate `GET /v1/models` (§11) |
| Attribution header | `anthropic-workspace`, not `OpenAI-Project` |
| `count_tokens` | On mantle for the current generation; on `bedrock-runtime` only for Claude **4.x**, and it refuses inference profiles (§8b) |
| Streaming shape | Named SSE events, not `data: {…}` frames |
| `content[0]` | Reasoning models lead with a `thinking` block — filter by type |
| Retention gating | A model can be `unavailable` under your mode; check `allowed_modes` |

## Next
- `02-thinking-tools-and-caching.ipynb` — adaptive thinking, tool loops, prompt
  caching
- `03-agentic-computer-use-and-memory.ipynb` — computer use, memory, compaction
- Other families: `../01-openai-gpt/`, `../03-google-gemma/`, `../04-qwen/`

## Also on `bedrock-runtime`? Claude

Claude is the one family where the runtime path needs an **inference profile**. Almost every Claude model is `INFERENCE_PROFILE`-only, so a bare model ID is rejected outright and you must use the `us.`-prefixed form.

`endpoints_for()` asks both catalogues rather than trusting a table, so the cell
below tells you today's answer. Converse is worth reaching for when you want one
request shape across providers, or a feature that only `bedrock-runtime` carries.


In [24]:
from bedrock import (
    converse,
    converse_reasoning,
    endpoints_for,
    resolve_runtime_id,
)

MANTLE_ID = "anthropic.claude-sonnet-5"
RUNTIME_ID = "anthropic.claude-sonnet-5"

print("endpoint availability:", endpoints_for(MANTLE_ID))
print("mantle model id :", MANTLE_ID)
print("runtime model id:", RUNTIME_ID)
resolved = resolve_runtime_id(RUNTIME_ID)
print("converse sends  :", resolved)
if resolved != RUNTIME_ID:
    print("                  ^ resolved for you; the form above would be rejected")

# The same question, through Converse. Note the shape: content is a LIST of
# blocks rather than a string, and the token budget lives in inferenceConfig.
# The budget is generous on purpose - a reasoning model spends it on the trace
# first and returns no text block at all if it runs out.
text, response = converse(
    RUNTIME_ID,
    [
        {
            "role": "user",
            "content": [
                {"text": "Name one benefit of idempotency. One sentence."}
            ],
        }
    ],
    max_tokens=400,
    system="You are terse.",
)

error = (response.get("error") or {}).get("message")
if error:
    print("\ncall failed:", error[:200])
else:
    reasoning = converse_reasoning(response)
    print("\nstop reason:", response.get("stopReason"))
    print("tokens     :", response.get("usage", {}).get("totalTokens"))
    if reasoning:
        print(f"reasoning  : {len(reasoning)} chars (returned in a "
              "reasoningContent block, before the text)")
    if text.strip():
        print("answer     :", text.strip()[:200])
    else:
        # Empty text is NOT the same as a failed call. Say which it is.
        print("answer     : (none - the budget went to reasoning; raise max_tokens)")


endpoint availability: {'mantle': True, 'runtime': True}
mantle model id : anthropic.claude-sonnet-5
runtime model id: anthropic.claude-sonnet-5
converse sends  : us.anthropic.claude-sonnet-5
                  ^ resolved for you; the form above would be rejected



stop reason: end_turn
tokens     : 77
answer     : Idempotency lets you safely retry a failed or duplicate request without causing unintended side effects (e.g., double-charging or duplicate records).


## Converse in earnest — the tool loop, provider parameters, and caching

The earlier endpoint section proved this model answers through Converse. That is
the easy part. This section does the three things you actually need on
`bedrock-runtime`, because each differs from the `bedrock-mantle` equivalent:

1. **A complete tool round trip** — `toolUse` out, `toolResult` back in. Getting a
   tool *call* is half the job; feeding the result back is where the shapes bite.
2. **`additionalModelRequestFields`** — Converse normalises the common fields, so
   anything provider-specific goes through this escape hatch.
3. **`cachePoint`** — prompt caching is a first-class Converse block, and support
   for it is per model rather than universal.


In [25]:
from bedrock import converse_text, converse_tool_uses, resolve_runtime_id, runtime_client

RUNTIME_ID = "anthropic.claude-sonnet-5"
runtime = runtime_client(REGION)
resolved = resolve_runtime_id(RUNTIME_ID, REGION)

# Converse tool shape: toolSpec, and the JSON Schema nests under inputSchema.json.
# This is NOT the OpenAI shape - there is no {"type": "function"} wrapper.
WEATHER_TOOL = {
    "toolSpec": {
        "name": "get_weather",
        "description": "Current weather for a city",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
            }
        },
    }
}

history = [
    {"role": "user", "content": [{"text": "What is the weather in Singapore? Use the tool."}]}
]
first = runtime.converse(
    modelId=resolved,
    messages=history,
    toolConfig={"tools": [WEATHER_TOOL]},
    inferenceConfig={"maxTokens": 500},
)
print("turn 1 stop reason:", first.get("stopReason"))
print("turn 1 blocks     :", [next(iter(b)) for b in first["output"]["message"]["content"]])

uses = converse_tool_uses(first)
if not uses:
    print("no tool call this run - tool_choice defaults to the model's discretion;")
    print("retry, or set toolConfig['toolChoice'] to compel one.")
else:
    use = uses[0]
    print(f"tool call         : {use['name']}({use['input']})")
    # Validate before acting on it. A malformed call still reports tool_use.
    city = str(use["input"].get("city", "")).lower()
    print("arguments valid   :", "yes" if "singapore" in city else f"NO ({use['input']})")

    # Echo the assistant turn back VERBATIM, then answer with a toolResult whose
    # toolUseId matches. Dropping either breaks the loop with a 400.
    history.append(first["output"]["message"])
    history.append(
        {
            "role": "user",
            "content": [
                {
                    "toolResult": {
                        "toolUseId": use["toolUseId"],
                        "content": [{"json": {"tempC": 31, "conditions": "humid"}}],
                    }
                }
            ],
        }
    )
    second = runtime.converse(
        modelId=resolved,
        messages=history,
        toolConfig={"tools": [WEATHER_TOOL]},
        inferenceConfig={"maxTokens": 300},
    )
    print("turn 2 stop reason:", second.get("stopReason"))
    print("final answer      :", converse_text(second).strip()[:160])


turn 1 stop reason: tool_use
turn 1 blocks     : ['toolUse']
tool call         : get_weather({'city': 'Singapore'})
arguments valid   : yes


turn 2 stop reason: end_turn
final answer      : The current weather in Singapore is **humid** with a temperature of **31°C**.


In [26]:
# Converse normalises maxTokens, temperature, topP and stopSequences. Anything
# provider-specific goes through additionalModelRequestFields, unvalidated by
# Converse and passed to the provider as-is. That makes it powerful and sharp:
# a key this model does not recognise is a 400, not a silent no-op.
from bedrock import converse_reasoning

PUZZLE = (
    "A bat and ball cost $1.10 together. The bat costs $1.00 more than the ball. "
    "How much is the ball?"
)

for label, extra in [
    ("no extra fields", None),
    ("provider fields", {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}}),
]:
    kwargs = {"additionalModelRequestFields": extra} if extra else {}
    try:
        response = runtime.converse(
            modelId=resolved,
            messages=[{"role": "user", "content": [{"text": PUZZLE}]}],
            inferenceConfig={"maxTokens": 900},
            **kwargs,
        )
    except Exception as exc:
        print(f"{label:<16} {type(exc).__name__}: {str(exc)[-90:]}")
        continue
    blocks = [next(iter(b)) for b in response["output"]["message"]["content"]]
    trace = converse_reasoning(response)
    answer = converse_text(response).strip().replace("\n", " ")
    print(f"{label:<16} out={response['usage']['outputTokens']:>4} blocks={blocks}")
    print(f"{'':<16} reasoning={len(trace)} chars | {answer[:70]}")

print()
print("Note whether a reasoningContent block appears above. Some models return the")
print("trace as a typed block on Converse and some do not, so read the blocks")
print("rather than assuming - and never index content[0].")


no extra fields  out= 326 blocks=['text']
                 reasoning=0 chars | # Bat and Ball Problem  **The ball costs $0.05 (5 cents)**  ## The Mat


provider fields  out= 169 blocks=['text']
                 reasoning=0 chars | The ball costs **$0.05** (5 cents).  Here's the math: If the ball cost

Note whether a reasoningContent block appears above. Some models return the
trace as a typed block on Converse and some do not, so read the blocks
rather than assuming - and never index content[0].


In [27]:
# cachePoint marks a prefix as cacheable. Everything BEFORE the marker is cached;
# the marker goes last in the block list it applies to. Watch the usage fields:
# the first call writes, the second reads.
HANDBOOK = "You are a support handbook. " + (
    "Retries: use exponential backoff with full jitter, cap at 16 seconds. " * 160
)


def cached_call():
    return runtime.converse(
        modelId=resolved,
        system=[{"text": HANDBOOK}, {"cachePoint": {"type": "default"}}],
        messages=[{"role": "user", "content": [{"text": "One line: the retry policy?"}]}],
        inferenceConfig={"maxTokens": 60},
    )


print(f"{'call':<6} {'write':>8} {'read':>8}  total")
print("-" * 40)
for n in (1, 2):
    usage = cached_call()["usage"]
    print(
        f"{n:<6} {str(usage.get('cacheWriteInputTokens')):>8} "
        f"{str(usage.get('cacheReadInputTokens')):>8}  {usage['totalTokens']}"
    )

print()
print("Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a")
print("lower rate than fresh input, so a long stable system prompt reused across")
print("many calls is where this pays. Check the pricing page for the current ratio.")


call      write     read  total
----------------------------------------


1          4332        0  4370


2          None     4332  4370

Call 1 writes the prefix, call 2 reads it back. Cached input is billed at a
lower rate than fresh input, so a long stable system prompt reused across
many calls is where this pays. Check the pricing page for the current ratio.


### What this section adds over the endpoint check above

- **The tool loop is the part that bites.** `toolSpec` is not the OpenAI shape,
  the JSON Schema nests under `inputSchema.json`, and the second turn must echo the
  assistant message back verbatim alongside a `toolResult` whose `toolUseId`
  matches. Miss any of that and you get a 400.
- **`additionalModelRequestFields` is unvalidated by Converse.** It is the only way
  to reach provider-specific behaviour, and a key the model does not recognise
  fails the call rather than being ignored.
- **Feature support is per model, not per endpoint.** Read the output above rather
  than carrying an assumption over from another family.
